# 4 — Validación end-to-end

Esta tarea convierte las expectativas en controles ejecutables. Si una condición falla, el Job queda en rojo. También registra métricas por corrida para demostrar que reprocesar el mismo lote no duplica resultados.

In [0]:
dbutils.widgets.text("student_id", "alumno01")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"])
dbutils.widgets.text("expected_batch_id", "batch_002")
dbutils.widgets.text("job_run_id", "interactive")

In [0]:
%run ../common/config

In [0]:
from decimal import Decimal
import re
from pyspark.sql import functions as F

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
expected = dbutils.widgets.get("expected_batch_id").strip().lower()
assert re.fullmatch(r"batch_[0-9]{3,6}", expected), "expected_batch_id debe tener formato batch_002"
job_run_id = dbutils.widgets.get("job_run_id").strip()[:100] or "interactive"
names = {name: qualified_table(config, name) for name in [
    "bronze_transactions_incremental", "silver_transactions", "silver_transactions_quarantine",
    "silver_customers", "silver_products", "gold_daily_sales",
    "gold_customer_risk", "gold_batch_summary", "pipeline_run_audit"]}
missing = [name for name, table in names.items() if name != "pipeline_run_audit" and not spark.catalog.tableExists(table)]
assert not missing, f"Faltan salidas del pipeline: {missing}"

In [0]:
tx = spark.table(names["silver_transactions"])
quarantine = spark.table(names["silver_transactions_quarantine"])
daily = spark.table(names["gold_daily_sales"])
batch = spark.table(names["gold_batch_summary"])
silver_metrics = tx.agg(F.count("*").alias("rows"), F.countDistinct("transaction_id").alias("distinct_ids"), F.sum("amount").cast("decimal(18,2)").alias("amount")).first()
gold_metrics = daily.agg(F.sum("transaction_count").alias("rows"), F.sum("total_amount").cast("decimal(18,2)").alias("amount")).first()
batch_metrics = batch.where(F.col("source_batch_id") == expected).first()
bronze_batch_rows = spark.table(names["bronze_transactions_incremental"]).where(F.col("source_batch_id") == expected).count()
silver_batch_rows = tx.where(F.col("source_batch_id") == expected).count()
rejected_batch_rows = quarantine.where(F.col("source_batch_id") == expected).count()
correction = tx.where(F.col("transaction_id") == 42).select("amount", "source_batch_id").first()
checks = {
    "silver_no_duplicate_ids": silver_metrics.rows == silver_metrics.distinct_ids,
    "gold_reconciles_rows": gold_metrics.rows == silver_metrics.rows,
    "gold_reconciles_amount": gold_metrics.amount == silver_metrics.amount,
    "batch_arrived_in_bronze": bronze_batch_rows > 0,
    "batch_arrived_in_silver": silver_batch_rows > 0,
    "batch_has_controlled_rejections": rejected_batch_rows >= 2,
    "batch_is_visible_in_gold": batch_metrics is not None and batch_metrics.accepted_transactions == silver_batch_rows and batch_metrics.rejected_transactions == rejected_batch_rows,
    "historical_correction_applied": correction is not None and correction.amount == Decimal("1999.99") and correction.source_batch_id == expected,
}
display(spark.createDataFrame([(name, passed) for name, passed in checks.items()], ["control", "passed"]))
failed = [name for name, passed in checks.items() if not passed]
assert not failed, f"Fallaron controles: {failed}"

control,passed
silver_no_duplicate_ids,true
gold_reconciles_rows,true
gold_reconciles_amount,true
batch_arrived_in_bronze,true
batch_arrived_in_silver,true
batch_has_controlled_rejections,true
batch_is_visible_in_gold,true
historical_correction_applied,false


---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
File <command-8730009665555718>, line 24
     22 display(spark.createDataFrame([(name, passed) for name, passed in checks.items()], ["control", "passed"]))
     23 failed = [name for name, passed in checks.items() if not passed]
---> 24 assert not failed, f"Fallaron controles: {failed}"

AssertionError: Fallaron controles: ['historical_correction_applied']

In [0]:
audit = names["pipeline_run_audit"]
spark.sql(f"""CREATE TABLE IF NOT EXISTS {audit} (
 job_run_id STRING, expected_batch_id STRING, silver_rows BIGINT, quarantine_rows BIGINT,
 gold_rows BIGINT, gold_total_amount DECIMAL(18,2), recorded_at TIMESTAMP, idempotence_compared BOOLEAN
) USING DELTA""")
prior = (spark.table(audit).orderBy(F.col("recorded_at").desc())
    .select("expected_batch_id", "silver_rows", "quarantine_rows", "gold_rows", "gold_total_amount").first())
comparable = prior is not None and prior.expected_batch_id == expected
current_metrics = (silver_metrics.rows, quarantine.count(), daily.count(), gold_metrics.amount)
if comparable:
    prior_metrics = (prior.silver_rows, prior.quarantine_rows, prior.gold_rows, prior.gold_total_amount)
    assert current_metrics == prior_metrics, f"La reejecución cambió métricas: antes={prior_metrics}, ahora={current_metrics}"
audit_row = spark.range(1).select(
    F.lit(job_run_id).alias("job_run_id"), F.lit(expected).alias("expected_batch_id"),
    F.lit(current_metrics[0]).cast("long").alias("silver_rows"),
    F.lit(current_metrics[1]).cast("long").alias("quarantine_rows"),
    F.lit(current_metrics[2]).cast("long").alias("gold_rows"),
    F.lit(current_metrics[3]).cast("decimal(18,2)").alias("gold_total_amount"),
    F.current_timestamp().alias("recorded_at"), F.lit(comparable).alias("idempotence_compared"))
audit_row.write.mode("append").saveAsTable(audit)
display(spark.table(audit).orderBy(F.col("recorded_at").desc()).limit(10))
print("Validación completa. Reejecución consecutiva comparada:", comparable)

job_run_id,expected_batch_id,silver_rows,quarantine_rows,gold_rows,gold_total_amount,recorded_at,idempotence_compared
interactive,batch_002,50349,55,669,50431198.33,2026-10-05T21:55:45.163Z,false
114132714324517,batch_003,50349,55,669,50431198.33,2026-10-03T21:38:22.075Z,false
508735289211689,batch_002,50149,53,598,50286183.33,2026-10-03T21:14:33.514Z,true
1114417553147735,batch_002,50149,53,598,50286183.33,2026-10-03T20:34:30.115Z,false


Validación completa. Reejecución consecutiva comparada: False
